# Python and TypeScript external functions

Host callbacks from real notebook cells: top-level await, blocking calls, and worker threads. No event-loop patching.

In [ ]:
import asyncio
import contextvars
import threading
from bashkit import BashTool

request_id = contextvars.ContextVar("request_id", default="missing")
caller_loop = asyncio.get_running_loop()

async def handler(name, args, kwargs):
    await asyncio.sleep(0)
    return {"value": args[0] * kwargs["multiplier"], "request": request_id.get(),
            "caller_loop": asyncio.get_running_loop() is caller_loop}

tool = BashTool(python=True, env={"BASHKIT_ALLOW_INPROCESS_PYTHON": "1"},
                external_functions=["lookup"], external_handler=handler)
script = "python -c 'r = lookup(21, multiplier=2); print(r[\"value\"], r[\"request\"], r[\"caller_loop\"])'"
request_id.set("async-cell")
result = await tool.execute(script)
assert result.exit_code == 0, result.stderr
assert result.stdout == "42 async-cell True\n"
print(result.stdout, end="")

In [ ]:
# A synchronous call inside the notebook loop uses an independent worker loop.
request_id.set("sync-cell")
result = tool.execute_sync(script)
assert result.exit_code == 0, result.stderr
assert result.stdout == "42 sync-cell False\n"
print(result.stdout, end="")

In [ ]:
# Reusing the same tool from another thread does not retain the prior caller loop.
def worker():
    async def run():
        request_id.set("worker-cell")
        return await tool.execute(script)
    return asyncio.run(run())

result = await asyncio.to_thread(worker)
assert result.exit_code == 0, result.stderr
assert result.stdout == "42 worker-cell False\n"
tool.reset()
request_id.set("after-reset")
result = await tool.execute(script)
assert result.stdout == "42 after-reset True\n", result.stderr
print("NOTEBOOK_EXTERNAL_FUNCTIONS_OK")

In [ ]:
# TypeScript functions receive (name, args); JSON objects carry named options.
async def ts_handler(name, args):
    await asyncio.sleep(0)
    return {"value": args[0] * args[1]["multiplier"], "request": request_id.get(),
            "caller_loop": asyncio.get_running_loop() is caller_loop}

ts_tool = BashTool(typescript=True, typescript_external_functions=["lookup"],
                   typescript_external_handler=ts_handler)
# Return the final expression: upstream ZapCode loses console.log output after suspension.
ts_script = "ts -c 'const r = await lookup(21, {multiplier: 2}); [r.value, r.request, r.caller_loop]'"
request_id.set("ts-async-cell")
result = await ts_tool.execute(ts_script)
assert result.exit_code == 0, result.stderr
assert result.stdout == "42,ts-async-cell,true\n"
print(result.stdout, end="")

In [ ]:
request_id.set("ts-sync-cell")
result = ts_tool.execute_sync(ts_script)
assert result.exit_code == 0, result.stderr
assert result.stdout == "42,ts-sync-cell,false\n"
ts_tool.reset()
request_id.set("ts-after-reset")
result = await ts_tool.execute(ts_script)
assert result.stdout == "42,ts-after-reset,true\n", result.stderr
print("NOTEBOOK_TYPESCRIPT_EXTERNAL_FUNCTIONS_OK")